In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow

############################################################################################################
########## It will be asked for your huggingface access token to load qwen packages ########################
############################################################################################################



In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

MODEL_NAME = "google/gemma-3-4b-it"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "India"    # cultural context for classification - India, China, Ireland

IMAGE_DIR  = "sample/"   # folder containing meme images
INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
OUTPUT_DIR = "results/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )